In [ ]:
# ============================================
# 딥러닝 훈련 기술 정리 (Colab 주석 버전)
# ============================================

# ------------------------------------------------
# 1. 배치 정규화 (Batch Normalization)
# ------------------------------------------------
# [역할]
# - 미니배치 단위로 평균/분산을 정규화하여 신경망 내부의 분포 변화를 줄임
#   → 학습을 안정화하고, 큰 Learning Rate 사용 가능
# - 약한 정규화 효과로 과적합을 완화

# [사용법]
# - CNN: Conv → BatchNorm → ReLU 순서가 기본 패턴
# - Transformer 계열은 LayerNorm을 주로 사용하지만 BN도 구조에 따라 사용 가능
# - PyTorch 예:
#     nn.BatchNorm2d(num_features)

# [주의점]
# - 배치 크기가 작으면(batch < 8 등) 통계량이 불안정해 품질이 떨어질 수 있음.


# ------------------------------------------------
# 2. 학습률 스케줄링 (Learning Rate Scheduling)
# ------------------------------------------------
# [역할]
# - 학습률을 동적으로 조절하여 빠른 수렴 + 안정적인 최적화 성능 확보
# - 초기에는 크게 → 후반에는 작게 (전형적인 패턴)
# - 지역 최소점 탈출 및 진동 억제에 도움

# [주요 스케줄러]
# - StepLR: 일정 epoch마다 감소
# - ExponentialLR: 지수 감소
# - CosineAnnealingLR: Cosine 곡선 형태로 감소
# - OneCycleLR: 학습률을 빠르게 올렸다가 천천히 줄이는 방식
# - Warmup: 초기 학습률을 아주 낮게 시작해 서서히 증가 (Transformer 표준)

# [PyTorch 예]
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
# for epoch in range(100):
#     train()
#     scheduler.step()


# ------------------------------------------------
# 3. 데이터 증강 (Data Augmentation)
# ------------------------------------------------
# [역할]
# - 데이터 다양성을 증가시켜 과적합을 방지하고 일반화 성능을 향상
# - 카메라 각도·조명·노이즈 등 현실 변화를 인공적으로 시뮬레이션

# [기본적 증강 기법]
# - RandomHorizontalFlip, RandomCrop, RandomRotation
# - ColorJitter(밝기/대비/채도 변형)
# - Gaussian Noise
# - Cutout / Mixup / CutMix 등 고급 기법

# [PyTorch 예]
# transform = transforms.Compose([
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomCrop(32, padding=4),
#     transforms.ToTensor()
# ])

# [주의점]
# - Validation/Test set에는 절대 증강을 적용하지 않음.


# ------------------------------------------------
# 4. 가중치 초기화 (Weight Initialization)
# ------------------------------------------------
# [역할]
# - 기울기 폭발/소실을 방지하여 학습 안정성을 확보
# - 적절한 초기화는 학습 속도와 수렴 성능을 크게 향상

# [대표 초기화 방식]
# - Xavier 초기화 (tanh, sigmoid 등)
#     nn.init.xavier_uniform_(layer.weight)
# - He(Kaiming) 초기화 (ReLU 기반 신경망 표준)
#     nn.init.kaiming_normal_(layer.weight, mode='fan_in')
# - Bias는 대부분 0 초기화

# [사용 시 주의]
# - 프레임워크의 기본 초기화도 대체로 충분하나,
#   Residual 구조, GAN 등에서는 명시적으로 설정하는 경우가 많음.


# ------------------------------------------------
# 5. 정규화 기법 (Regularization)
# ------------------------------------------------
# [역할]
# - 과적합을 방지하고 일반화 성능을 향상시키는 기법 전반

# [주요 기법]
# 1) L2 Regularization(Weight Decay)
#     optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 2) Dropout
# - 특정 확률로 뉴런을 비활성화하여 특정 경로에 대한 과도한 의존 방지

# 3) Early Stopping
# - 검증 손실이 더 이상 감소하지 않으면 학습 중단 → 과적합 방지


# ------------------------------------------------
# 6. 최적화 알고리즘 (Optimization Algorithms)
# ------------------------------------------------
# [역할]
# - 손실 함수를 최소화하는 방향으로 파라미터를 갱신

# [대표 예시]
# - SGD + Momentum: 고전적이지만 여전히 강력함
# - Adam: 적응적 학습률로 빠른 수렴
# - AdamW: Weight decay 적용 방식 개선 (Transformer 표준)

# [PyTorch 예]
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


# ------------------------------------------------
# 7. Gradient Clipping
# ------------------------------------------------
# [역할]
# - 기울기 폭발 방지(NLP, RNN, Transformer에서 중요)

# [예]
# torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)


# ------------------------------------------------
# 8. Mixed Precision Training (반정밀도 학습)
# ------------------------------------------------
# [역할]
# - GPU 메모리 절약 + 학습 속도 증가 (FP16 사용)
# - 대규모 모델 학습 시 표준 기술로 자리잡음

# [사용 예]
# from torch.cuda.amp import GradScaler, autocast


# ------------------------------------------------
# 9. Warmup (학습률 워밍업)
# ------------------------------------------------
# [역할]
# - 초기 학습률을 매우 낮게 설정해 안정적으로 훈련 시작
# - Transformer, ViT, BERT, GPT 등에서 사실상 필수

# [의미]
# - 초기에 weight가 불안정할 때 너무 큰 LR을 사용하면 발산 가능
#   → 몇 천 step 동안 LR을 점진적으로 증가시켜 해결


# ------------------------------------------------
# 10. Label Smoothing
# ------------------------------------------------
# [역할]
# - 정답 라벨을 1.0이 아닌 0.9 등으로 완화하여 모델의 과도한 확신을 줄임
# - 분류 문제에서 일반화 성능 향상

# [예]
# loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
